# 10 - Hyperparamter Search
Random search for unfrozen EVA fine-tuning with `ArcFaceModel`. Each sampled run trains independently and logs metrics to W&B.

In [ ]:
import random
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.wandb_utils as wandb_utils
from src.models import ArcFaceModel
from src.training import build_optimizer, fit
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

RANDOM_SEED = 42
SEARCH_SEED = 105
set_seed(RANDOM_SEED)

device = get_device()
device

device(type='cuda')

## Base Config

In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("../checkpoints/e10_hyperparamter_search"),
    "results_path": Path("../checkpoints/e10_hyperparamter_search/random_search_results.csv"),

    # Backbone
    "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
    "input_size": 448,
    "freeze_backbone": False,

    # Model defaults
    "embedding_dim": 256,
    "hidden_dim": 512,
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training defaults
    "batch_size": 32,
    "learning_rate": 1e-4,
    "head_learning_rate": 1e-4,
    "backbone_learning_rate": 1e-5,
    "weight_decay": 1e-4,
    "num_epochs": 25,
    "patience": 8,
    "scheduler_patience": 2,
    "val_split": 0.2,
    "num_workers": 2,
    "augment_train": True,

    # Reproducibility
    "seed": RANDOM_SEED,
    "search_seed": SEARCH_SEED,

    # Image normalization
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

config["rerank"] = {
    "enabled": True,
    "k1": 20,
    "k2": 6,
    "lambda_value": 0.3,
}

config["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
config

{'data_dir': PosixPath('../data'),
 'checkpoint_dir': PosixPath('../checkpoints/e10_hyperparamter_search'),
 'results_path': PosixPath('../checkpoints/e10_hyperparamter_search/random_search_results.csv'),
 'backbone_name': 'eva02_large_patch14_448.mim_m38m_ft_in22k_in1k',
 'input_size': 448,
 'freeze_backbone': False,
 'embedding_dim': 256,
 'hidden_dim': 512,
 'arcface_margin': 0.5,
 'arcface_scale': 64.0,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0001,
 'head_learning_rate': 0.0001,
 'backbone_learning_rate': 1e-05,
 'weight_decay': 0.0001,
 'num_epochs': 2,
 'patience': 8,
 'scheduler_patience': 2,
 'val_split': 0.2,
 'num_workers': 2,
 'augment_train': True,
 'seed': 42,
 'search_seed': 100,
 'mean': (0.481, 0.457, 0.408),
 'std': (0.268, 0.261, 0.275),
 'rerank': {'enabled': True, 'k1': 20, 'k2': 6, 'lambda_value': 0.3}}

## Search Space

In [ ]:
search_space = {
    "head_learning_rate": [3e-5, 1e-4, 3e-4],
    "backbone_learning_rate": [3e-6, 1e-5, 3e-5],
    "weight_decay": [1e-5, 1e-4, 5e-4],
    "dropout": [0.2, 0.3, 0.4],
    "augment_train": [True,False],
    "batch_size": [16, 32],
    "embedding_dim": [256, 384],
    "hidden_dim": [512, 768],
    "patience": [8],
    "scheduler_patience": [2],
}

num_random_runs = 12
search_space

{'head_learning_rate': [3e-05, 0.0001, 0.0003],
 'backbone_learning_rate': [3e-06, 1e-05, 3e-05],
 'weight_decay': [1e-05, 0.0001, 0.0005],
 'dropout': [0.2, 0.3, 0.4],
 'augment_train': [True, False],
 'batch_size': [16, 32],
 'embedding_dim': [256, 384],
 'hidden_dim': [512, 768],
 'patience': [8],
 'scheduler_patience': [2]}

## Data Loading

In [7]:
def load_data(config, backbone):
    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    train_data, val_data = data.train_val_split(
        train_df,
        val_split=config["val_split"],
        seed=config["seed"],
        stratify_col="ground_truth",
    )

    train_loader, val_loader = data.create_dataloaders(
        train_data,
        val_data,
        img_dir=config["data_dir"] / "train" / "train",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        mean=config["mean"],
        std=config["std"],
        weighted_sampling=False,
        augment=config.get("augment_train", False),
    )

    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    return {
        "label_encoder": label_encoder,
        "num_classes": num_classes,
        "train_loader": train_loader,
        "val_loader": val_loader,
    }


## Random Search Utilities

In [8]:
def sample_random_experiments(search_space, num_runs, seed):
    rng = random.Random(seed)
    experiments = []
    seen = set()

    keys = list(search_space.keys())
    max_unique = 1
    for key in keys:
        max_unique *= len(search_space[key])

    if num_runs > max_unique:
        raise ValueError(f"Requested {num_runs} runs but only {max_unique} unique combinations exist")

    while len(experiments) < num_runs:
        sampled = {key: rng.choice(search_space[key]) for key in keys}
        signature = tuple((key, sampled[key]) for key in keys)
        if signature in seen:
            continue
        seen.add(signature)

        run_name = (
            f"eva_unfrozen_rs_{len(experiments)+1:02d}"
            f"_hlr{sampled['head_learning_rate']:.0e}"
            f"_blr{sampled['backbone_learning_rate']:.0e}"
            f"_wd{sampled['weight_decay']:.0e}"
            f"_do{sampled['dropout']:.1f}"
            f"_aug{int(sampled['augment_train'])}"
            f"_bs{sampled['batch_size']}"
        )

        sampled["name"] = run_name
        experiments.append(sampled)

    return experiments


experiments = sample_random_experiments(search_space, num_random_runs, config["search_seed"])
pd.DataFrame(experiments)

,head_learning_rate,backbone_learning_rate,weight_decay,dropout,augment_train,batch_size,embedding_dim,hidden_dim,patience,scheduler_patience,name
0,0.00003,0.00001,0.0001,0.2,False,32,384,512,8,2,eva_unfrozen_rs_01_hlr3e-05_blr1e-05_wd1e-04_d...


## Run Search

In [9]:
def run_experiment(experiment, run_idx, total_runs):
    run_config = config.copy()
    run_config.update(experiment)
    run_config["seed"] = config["seed"]
    run_config["freeze_backbone"] = False
    run_config["backbone_name"] = config["backbone_name"]
    run_config["input_size"] = config["input_size"]
    run_config["learning_rate"] = run_config["head_learning_rate"]

    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {run_config['name']}")
    print({key: run_config[key] for key in search_space})

    set_seed(run_config["seed"])

    train_df = data.load_train_df(run_config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    model = ArcFaceModel(
        backbone_name=run_config["backbone_name"],
        num_classes=num_classes,
        embedding_dim=run_config["embedding_dim"],
        hidden_dim=run_config["hidden_dim"],
        margin=run_config["arcface_margin"],
        scale=run_config["arcface_scale"],
        dropout=run_config["dropout"],
        pretrained=True,
        freeze_backbone=False,
    ).to(device)

    data_bundle = load_data(run_config, model.backbone)
    label_encoder = data_bundle["label_encoder"]
    num_classes = data_bundle["num_classes"]
    train_loader = data_bundle["train_loader"]
    val_loader = data_bundle["val_loader"]

    if model.arcface.num_classes != num_classes:
        raise ValueError(f"Model num_classes={model.arcface.num_classes} but dataset has {num_classes}")

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone_params = sum(p.numel() for p in model.backbone.parameters())

    wandb = wandb_utils.init_wandb(
        run_config,
        run_name=run_config["name"],
        param_count=trainable_params,
        param_count_backbone=backbone_params,
    )
    wandb.run.summary["total_param_count"] = total_params
    wandb.run.summary["trainable_param_count"] = trainable_params
    wandb.run.summary["search_run_index"] = run_idx
    wandb.run.summary["search_total_runs"] = total_runs

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, run_config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=run_config["scheduler_patience"],
    )

    checkpoint_name = f"{run_config['name']}.pth"
    checkpoint_path = run_config["checkpoint_dir"] / checkpoint_name
    results = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=run_config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_mAP_rerank"] = results.get("best_map_rerank")
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    wandb_utils.log_checkpoint_artifact(
        wandb,
        checkpoint_path,
        artifact_name=run_config["name"],
        description="random-search checkpoint",
    )

    wandb.finish()

    return {
        "name": run_config["name"],
        **{key: run_config[key] for key in search_space},
        "best_val_loss": results["best_val_loss"],
        "best_map": results["best_map"],
        "best_map_rerank": results.get("best_map_rerank"),
        "best_epoch": results["best_epoch"],
        "epochs_ran": results["epochs_ran"],
        "checkpoint_path": str(checkpoint_path),
    }


search_results = []
for run_idx, experiment in enumerate(experiments, start=1):
    result = run_experiment(experiment, run_idx, len(experiments))
    search_results.append(result)

results_df = pd.DataFrame(search_results).sort_values(["best_map_rerank", "best_map"], ascending=False)
results_df.to_csv(config["results_path"], index=False)
results_df

Run 1/1: eva_unfrozen_rs_01_hlr3e-05_blr1e-05_wd1e-04_do0.2_aug0_bs32
{'head_learning_rate': 3e-05, 'backbone_learning_rate': 1e-05, 'weight_decay': 0.0001, 'dropout': 0.2, 'augment_train': False, 'batch_size': 32, 'embedding_dim': 384, 'hidden_dim': 512, 'patience': 8, 'scheduler_patience': 2}


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /sc/home/maximilian.speer/.netrc


Train batches: 48 | Val batches: 12


wandb: Currently logged in as: maximilian-speer (maximilian-speer-hasso-plattner-institut) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training for 2 epochs...

Epoch 1/2


...

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

  Train Loss: 10.5052 | Train Acc: 25.1%
  Val Loss:   5.8919 | Val Acc:   62.8%
  Val mAP:    0.7984 | LR: 1.00e-05
  Val mAP RR: 0.7962 | k1=20 k2=6 lambda=0.3
  [New best model saved]

Training complete!
Best epoch: 2 (Val Loss: 5.8919, Val mAP: 0.7984)


epoch,▁█
learning_rate,▁▁
train_acc,▁█
train_loss,█▁
val_acc,▁█
val_loss,█▁
val_map,▁█
val_map_rerank,▁█
best_epoch,2
best_val_loss,5.89193
best_val_mAP,0.79836


,name,head_learning_rate,backbone_learning_rate,weight_decay,dropout,augment_train,batch_size,embedding_dim,hidden_dim,patience,scheduler_patience,best_val_loss,best_map,best_map_rerank,best_epoch,epochs_ran,checkpoint_path
0,eva_unfrozen_rs_01_hlr3e-05_blr1e-05_wd1e-04_d...,0.00003,0.00001,0.0001,0.2,False,32,384,512,8,2,5.891926,0.798359,0.796202,2,2,../checkpoints/e10_hyperparamter_search/eva_un...


## Best Runs

In [ ]:
results_df[[
    "name",
    "best_map",
    "best_map_rerank",
    "best_val_loss",
    "best_epoch",
    "head_learning_rate",
    "backbone_learning_rate",
    "weight_decay",
    "dropout",
    "augment_train",
    "batch_size",
    "embedding_dim",
    "hidden_dim",
    "patience",
    "scheduler_patience",
]].head(10)

View the runs on W&B: [W&B Run Group](https://wandb.ai/juggling-jaguars/jaguar-reid-jugglingjaguars/groups/Experiment-5-HyperparameterSearch) | [W&B Sweep](https://wandb.ai/juggling-jaguars/jaguar-reid-jugglingjaguars/sweeps/df5f8s4d)

![](../images/e5_sweep_graph.png)